# 03 — Silver → Gold: `fact_provider_service`

The workhorse fact at the original grain: one row per
(`npi × hcpcs_cd × place_of_srvc`). Intentionally **narrow** — `hcpcs_desc`
is NOT included here. Power BI and the hero marts join `dim_hcpcs` for the
description string, which keeps the 9.6M-row fact small and the Vertipaq
engine fast.

Partitioned by `state_abrvtn` (matches the Power BI state slicer) and
Z-ORDERed on `(hcpcs_cd, specialty)` for procedure/specialty drilldowns.

In [0]:
from pyspark.sql import SparkSession, functions as F

from utils.secrets import configure_adls_oauth
from utils.paths   import SILVER_PHYSICIAN, GOLD

spark = SparkSession.builder.appName("silver_to_gold_fact_provider_service").getOrCreate()
spark.conf.set("spark.sql.adaptive.enabled", "true")
configure_adls_oauth(spark, dbutils)
silver = spark.read.format("delta").load(SILVER_PHYSICIAN)

In [0]:
fact = silver.select(
    "service_sk",
    F.col("Rndrng_NPI").alias("npi"),
    F.col("Rndrng_Prvdr_Type").alias("specialty"),
    F.col("Provider_Tier").alias("provider_tier"),
    F.col("Rndrng_Prvdr_State_Abrvtn").alias("state_abrvtn"),
    F.col("RUCA_Int").alias("ruca_int"),
    F.col("RUCA_Bucket").alias("ruca_bucket"),
    F.col("Is_Rural").alias("is_rural"),
    F.col("Is_Participating").alias("is_participating"),
    F.col("Is_Org").alias("is_org"),
    F.col("HCPCS_Cd").alias("hcpcs_cd"),                # FK → dim_hcpcs
    F.col("Is_Drug").alias("is_drug"),
    F.col("Place_Of_Srvc").alias("place_of_srvc"),
    F.col("Is_Facility").alias("is_facility"),
    "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
    "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Pymt_Amt", "Avg_Mdcr_Stdzd_Amt",
    "Tot_Sbmtd_Chrg", "Tot_Mdcr_Alowd_Amt", "Tot_Mdcr_Pymt_Amt", "Tot_Mdcr_Stdzd_Amt",
    "Markup_Ratio", "Mdcr_Pymt_Rate", "Geo_Adj_Factor", "Geo_Premium", "Patient_Exposure",
)

In [0]:
(fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("state_abrvtn")
    .save(GOLD("fact_provider_service")))

print(f"fact_provider_service written → {GOLD('fact_provider_service')}")

fact_provider_service written → abfss://gold@sthealthcareplatdev.dfs.core.windows.net/fact_provider_service/


## OPTIMIZE + ZORDER on the highest-selectivity filter columns



In [0]:
spark.sql(
    f"OPTIMIZE delta.`{GOLD('fact_provider_service')}` ZORDER BY (hcpcs_cd, specialty)"
)
print("fact OPTIMIZE complete.")

fact OPTIMIZE complete.
